In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/Users/lunecid/Documents/Project/DatoryLab/Data/부산광역시_대규모점포 등록현황_20230630.csv', encoding='cp949')

In [ ]:
df.info()

In [8]:
df['행정동'] = df['도로명주소'].str.extract(r'\(([^()\s]*동[^()]*)\)')

In [ ]:
df['행정동']

In [ ]:
pip install geopandas

In [13]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [15]:
# 2. GeoDataFrame으로 변환 (위도, 경도 → Point 객체)
df['geometry'] = df.apply(lambda row: Point(row['경도'], row['위도']), axis=1)
gdf_mart = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# 3. SHP 파일 (부산 행정동) 불러오기
gdf_dong = gpd.read_file('/Users/lunecid/Downloads/hangjeongdong_pusan.geojson')  # 필요한 경우 EPSG 확인

# 4. CRS 맞추기 (위도경도와 동일하게 맞추기)
gdf_dong = gdf_dong.to_crs("EPSG:4326")

# 5. 공간 조인: 마트 위치가 어느 행정동에 속하는지
gdf_joined = gpd.sjoin(gdf_mart, gdf_dong, how="left", predicate='intersects')

In [ ]:
gdf_joined 

In [ ]:
gdf_joined.iloc[31]

In [42]:
gdf_joined['d1'] = gdf_joined['adm_nm'].str.extract(r'(\w+(동|읍|리|면))$')[0]
gdf_joined['행정동코드'] = gdf_joined['adm_cd']

In [ ]:
print(gdf_joined['d1'].to_string())

In [44]:
gdf_joined = gdf_joined.groupby('d1').size().reset_index(name='대형마트_개수')

In [ ]:
print(gdf_joined.to_string())

In [48]:
gdf_joined = gdf_joined.rename(columns={'d1': '행정동'})

In [ ]:
gdf_joined.head()

In [50]:
gdf_joined.to_csv("행정동_매핑_마트데이터.csv", index=False, encoding='utf-8-sig')